In [ ]:
import os

# List files in the current directory to help locate the NPI data file
# This line is commented out as it's not strictly necessary for the final output.
# print("Files in the current directory:")
# for root, dirs, files in os.walk('.'):
#     for file in files:
#         print(os.path.join(root, file))

In [ ]:
import requests
import pandas as pd

# Base URL for the NPI Registry API
base_url = "https://npiregistry.cms.hhs.gov/api/"

def fetch_npi_data(city, state, enumeration_type='NPI-1', limit=200):
    all_results = []
    offset = 0
    max_skip = 1000 # API allows skipping up to 1000 records

    while True:
        params = {
            'version': '2.1',
            'city': city,
            'state': state,
            'enumeration_type': enumeration_type,
            'limit': limit,
            'skip': offset
        }

        try:
            response = requests.get(base_url, params=params)
            response.raise_for_status()  # Raise an exception for HTTP errors
            data = response.json()

            if 'results' in data and data['results']:
                all_results.extend(data['results'])
                print(f"Fetched {len(data['results'])} records from offset {offset}.")
                # Check if there are more results to fetch
                # The API returns an empty list if there are no more results
                if len(data['results']) < limit or offset >= max_skip:
                    print("No more results or max skip reached. Stopping pagination.")
                    break
                offset += limit
            else:
                print(f"No results found for city={city}, state={state} at offset {offset}. Stopping.")
                break

            # Safety break to prevent infinite loops and respect API limits
            if len(all_results) >= 1200: # Max possible records with skip limit
                print("Reached maximum retrievable records (1200) through API pagination. Stopping.")
                break

        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            break
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            break

    return all_results

print("Fetching NPI data for physicians in Atlanta, Georgia...")
atlanta_physicians_data = fetch_npi_data(city='ATLANTA', state='GA', enumeration_type='NPI-1')

if atlanta_physicians_data:
    df_atlanta_physicians = pd.DataFrame(atlanta_physicians_data)
    print(f"Total unique physicians found: {len(df_atlanta_physicians)}")
    display(df_atlanta_physicians.head())
    display(df_atlanta_physicians.info())
else:
    print("Could not retrieve any NPI data for Atlanta, Georgia physicians.")

Fetching NPI data for physicians in Atlanta, Georgia...
Fetched 200 records from offset 0.
Fetched 200 records from offset 200.
Fetched 200 records from offset 400.
Fetched 200 records from offset 600.
Fetched 200 records from offset 800.
Fetched 200 records from offset 1000.
No more results or max skip reached. Stopping pagination.
Total unique physicians found: 1200


,addresses,basic,created_epoch,endpoints,enumeration_type,identifiers,last_updated_epoch,number,other_names,practiceLocations,taxonomies
0,[{'address_1': '2625 PIEDMONT RD NE STE 56-378...,"{'enumeration_date': '2019-09-21', 'first_name...",1569092791000,[],NPI-1,[],1569092791000,1710538772,[],[],"[{'code': '3747A0650X', 'desc': 'Technician, A..."
1,"[{'address_1': '1365 CLIFTON RD NE # B', 'addr...","{'credential': 'MD', 'enumeration_date': '2006...",1156997057000,[],NPI-1,[],1183947785000,1750491783,[],[],"[{'code': '207W00000X', 'desc': 'Ophthalmology..."
2,"[{'address_1': 'PO BOX 748613', 'address_purpo...","{'certification_date': '2025-11-05', 'credenti...",1242150191000,"[{'address_1': '541 Sunset Ln Ste 305', 'addre...",NPI-1,"[{'code': '01', 'desc': 'Other (non-Medicare)'...",1762378271000,1700010287,[],[],"[{'code': '207X00000X', 'desc': 'Orthopaedic S..."
3,"[{'address_1': '2220 N DRUID HILLS RD NE', 'ad...","{'certification_date': '2025-10-17', 'credenti...",1749578102000,[],NPI-1,[],1760738784000,1669367702,[],[],"[{'code': '363A00000X', 'desc': 'Physician Ass..."
4,"[{'address_1': '5669 PEACHTREE DUNWOODY RD', '...","{'credential': 'NP', 'enumeration_date': '2013...",1381509969000,[],NPI-1,[],1544658425000,1265869051,[],"[{'address_1': '5669 PEACHTREE DUNWOODY RD', '...","[{'code': '363LA2200X', 'desc': 'Nurse Practit..."


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   addresses           1200 non-null   object
 1   basic               1200 non-null   object
 2   created_epoch       1200 non-null   object
 3   endpoints           1200 non-null   object
 4   enumeration_type    1200 non-null   object
 5   identifiers         1200 non-null   object
 6   last_updated_epoch  1200 non-null   object
 7   number              1200 non-null   object
 8   other_names         1200 non-null   object
 9   practiceLocations   1200 non-null   object
 10  taxonomies          1200 non-null   object
dtypes: object(11)
memory usage: 103.3+ KB


None

In [9]:
import pandas as pd

# 1. Helper functions to safely extract address details and provider metadata
def get_primary_address_detail(addresses, key):
    if addresses and isinstance(addresses, list) and addresses[0] and isinstance(addresses[0], dict):
        return addresses[0].get(key)
    return None

def get_primary_taxonomy_description(taxonomies):
    if taxonomies and isinstance(taxonomies, list):
        for taxonomy in taxonomies:
            if isinstance(taxonomy, dict) and taxonomy.get('primary') == True:
                return taxonomy.get('desc')
    return None

def get_full_name(basic):
    if basic and isinstance(basic, dict):
        first_name = basic.get('first_name', '')
        last_name = basic.get('last_name', '')
        return f"{first_name} {last_name}".strip()
    return None

# 2. Extract address components, taxonomy, names, phone, and fax
df_atlanta_physicians['primary_practice_address_1'] = df_atlanta_physicians['addresses'].apply(lambda x: get_primary_address_detail(x, 'address_1'))
df_atlanta_physicians['primary_practice_address_2'] = df_atlanta_physicians['addresses'].apply(lambda x: get_primary_address_detail(x, 'address_2'))
df_atlanta_physicians['primary_practice_city'] = df_atlanta_physicians['addresses'].apply(lambda x: get_primary_address_detail(x, 'city'))
df_atlanta_physicians['primary_practice_state'] = df_atlanta_physicians['addresses'].apply(lambda x: get_primary_address_detail(x, 'state'))
df_atlanta_physicians['primary_practice_postal_code'] = df_atlanta_physicians['addresses'].apply(lambda x: get_primary_address_detail(x, 'postal_code'))
df_atlanta_physicians['primary_practice_phone'] = df_atlanta_physicians['addresses'].apply(lambda x: get_primary_address_detail(x, 'telephone_number'))
df_atlanta_physicians['primary_practice_fax'] = df_atlanta_physicians['addresses'].apply(lambda x: get_primary_address_detail(x, 'fax_number'))
df_atlanta_physicians['primary_taxonomy_description'] = df_atlanta_physicians['taxonomies'].apply(get_primary_taxonomy_description)
df_atlanta_physicians['provider_full_name'] = df_atlanta_physicians['basic'].apply(get_full_name)

# 3. Sort by professional proximity hierarchy
sort_cols = [
    'primary_practice_state',
    'primary_practice_city',
    'primary_practice_postal_code',
    'primary_practice_address_1',
    'primary_practice_address_2',
    'primary_taxonomy_description',
    'provider_full_name'
]

for col in sort_cols:
    if col in df_atlanta_physicians.columns:
        df_atlanta_physicians[col] = df_atlanta_physicians[col].fillna('')

df_atlanta_physicians_sorted = df_atlanta_physicians.sort_values(by=sort_cols).reset_index(drop=True)

# 4. Filter for Atlanta, GA
df_atlanta_physicians_filtered_sorted = df_atlanta_physicians_sorted[
    (df_atlanta_physicians_sorted['primary_practice_city'].str.upper() == 'ATLANTA') &
    (df_atlanta_physicians_sorted['primary_practice_state'].str.upper() == 'GA')
].reset_index(drop=True)

# 5. Format full practice address string
df_atlanta_physicians_filtered_sorted['full_practice_address'] = (
    df_atlanta_physicians_filtered_sorted['primary_practice_address_1'] + ', '
    + df_atlanta_physicians_filtered_sorted['primary_practice_address_2'].apply(lambda x: x + ', ' if x else '')
    + df_atlanta_physicians_filtered_sorted['primary_practice_city'] + ', '
    + df_atlanta_physicians_filtered_sorted['primary_practice_state'] + ' '
    + df_atlanta_physicians_filtered_sorted['primary_practice_postal_code']
)

# Removed print and display statements as they are intermediate outputs.


Number of unique primary practice addresses in Atlanta, GA: 476


,provider_full_name,primary_taxonomy_description,full_practice_address,primary_practice_phone,primary_practice_fax
0,NADDIA AHMED,Nurse Practitioner,"277 GA-74 STE 210, ATLANTA, GA 30269",None,None
1,TIMOTHY ADAMS,"Specialist/Technologist, Athletic Trainer","PO BOX 3975, ATLANTA, GA 303023975",404-413-4177,None
2,KHALIL ABDULRAUF,Student in an Organized Health Care Education/...,"49 JESSE HILL JR DR, ATLANTA, GA 30303",404-251-8796,None
3,JANICE ABRAHAM,Anesthesiologist Assistant,"80 JESSE HILL DR SE, ATLANTA, GA 30303",404-616-1000,None
4,GLORIA AKLY,Registered Nurse,"99 JESSE HILL JR DRIVE SE, ROOM 402, ATLANTA, ...",None,None
5,TERESA ADES,"Nurse Practitioner, Family","250 WILLIAMS ST NW, 6TH FLOOR, ATLANTA, GA 303...",404-329-7785,404-327-6404
6,JENNIFER AKINS,Behavior Technician,"260 PEACHTREE ST NW STE 2200, ATLANTA, GA 3030...",855-832-6727,772-675-9100
7,SABRINA ALEXANDER,Behavior Technician,"260 PEACHTREE ST NW STE 2200, ATLANTA, GA 3030...",877-418-2978,866-500-2186
8,SCONDRIA AKERELE,Behavior Technician,"260 PEACHTREE ST NW STE 2200, ATLANTA, GA 3030...",706-330-8850,None
9,UGOMMA ABANUKAM,"Nurse Practitioner, Family","57 FORSYTH ST NW, ATLANTA, GA 303032229",None,None


Here's how you can download the full processed dataset (`df_atlanta_physicians_filtered_sorted`) to a CSV file:

In [20]:
from google.colab import files

# Define the filename for the downloaded CSV
output_filename = 'atlanta_physicians_processed_data.csv'

# Save the full processed DataFrame to a CSV file
df_atlanta_physicians_filtered_sorted.to_csv(output_filename, index=False)

print(f"\nYour processed data has been saved to '{output_filename}'.")
print("You can now download this file:")

# Provide a download link for the file
files.download(output_filename)


Your processed data has been saved to 'atlanta_physicians_processed_data.csv'.
You can now download this file:


,provider_full_name,primary_taxonomy_description,full_practice_address,primary_practice_phone,primary_practice_fax
0,NADDIA AHMED,Nurse Practitioner,"277 GA-74 STE 210, ATLANTA, GA 30269",None,None
1,TIMOTHY ADAMS,"Specialist/Technologist, Athletic Trainer","PO BOX 3975, ATLANTA, GA 303023975",404-413-4177,None
2,KHALIL ABDULRAUF,Student in an Organized Health Care Education/...,"49 JESSE HILL JR DR, ATLANTA, GA 30303",404-251-8796,None
3,JANICE ABRAHAM,Anesthesiologist Assistant,"80 JESSE HILL DR SE, ATLANTA, GA 30303",404-616-1000,None
4,GLORIA AKLY,Registered Nurse,"99 JESSE HILL JR DRIVE SE, ROOM 402, ATLANTA, ...",None,None
5,TERESA ADES,"Nurse Practitioner, Family","250 WILLIAMS ST NW, 6TH FLOOR, ATLANTA, GA 303...",404-329-7785,404-327-6404
6,JENNIFER AKINS,Behavior Technician,"260 PEACHTREE ST NW STE 2200, ATLANTA, GA 3030...",855-832-6727,772-675-9100
7,SABRINA ALEXANDER,Behavior Technician,"260 PEACHTREE ST NW STE 2200, ATLANTA, GA 3030...",877-418-2978,866-500-2186
8,SCONDRIA AKERELE,Behavior Technician,"260 PEACHTREE ST NW STE 2200, ATLANTA, GA 3030...",706-330-8850,None
9,UGOMMA ABANUKAM,"Nurse Practitioner, Family","57 FORSYTH ST NW, ATLANTA, GA 303032229",None,None


Total providers in dataset: 826
Missing primary practice phone: 132
Missing primary practice fax: 573
Missing BOTH phone and fax: 132


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
import json

# 1. Filter out providers missing either phone OR fax
df_complete = df_atlanta_physicians_filtered_sorted.dropna(
    subset=['primary_practice_phone', 'primary_practice_fax']
)

# Optional: If empty strings exist in phone or fax, filter them as well
df_complete = df_complete[
    (df_complete['primary_practice_phone'] != '') &
    (df_complete['primary_practice_fax'] != '')
]

# 2. Select relevant columns for the final JSON output
columns_to_export = [
    'provider_full_name',
    'primary_taxonomy_description',
    'full_practice_address',
    'primary_practice_phone',
    'primary_practice_fax'
]

# 3. Export to a JSON file
json_filename = 'atlanta_physicians_contact.json'
df_complete[columns_to_export].to_json(
    json_filename,
    orient='records',
    indent=4
)

print(f"Exported {len(df_complete)} provider records to {json_filename}")

# 4. Trigger download in Google Colab
from google.colab import files
files.download(json_filename)

Exported 253 provider records to atlanta_physicians_contact.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# This cell performed an analysis step that is not directly required for generating the requested CSV and JSON files, and has been removed as part of code cleanup.


Physicians sharing the same primary practice address (sample groups):

--- Address: 260 PEACHTREE ST NW STE 2200, ATLANTA, GA 303031292 ---


,provider_full_name,primary_taxonomy_description,primary_practice_address_1,primary_practice_city,primary_practice_postal_code
6,JENNIFER AKINS,Behavior Technician,260 PEACHTREE ST NW STE 2200,ATLANTA,303031292
7,SABRINA ALEXANDER,Behavior Technician,260 PEACHTREE ST NW STE 2200,ATLANTA,303031292
8,SCONDRIA AKERELE,Behavior Technician,260 PEACHTREE ST NW STE 2200,ATLANTA,303031292




--- Address: 80 JESSE HILL JR DR SE, ATLANTA, GA 303033031 ---


,provider_full_name,primary_taxonomy_description,primary_practice_address_1,primary_practice_city,primary_practice_postal_code
11,KARLA ALEMAN,"Dietitian, Registered",80 JESSE HILL JR DR SE,ATLANTA,303033031
12,MAAZ AHMED,Internal Medicine,80 JESSE HILL JR DR SE,ATLANTA,303033031
13,EVELYN ADESANYA,"Nurse Practitioner, Family",80 JESSE HILL JR DR SE,ATLANTA,303033031
14,MONSURAT ADEBAYO,"Nurse Practitioner, Family",80 JESSE HILL JR DR SE,ATLANTA,303033031
15,EDDY ALEMAN,Physician Assistant,80 JESSE HILL JR DR SE,ATLANTA,303033031
16,OLUFUNMI ADEWOYIN,"Registered Nurse, Critical Care Medicine",80 JESSE HILL JR DR SE,ATLANTA,303033031




--- Address: 49 JESSE HILL JR DR SE, ATLANTA, GA 303033049 ---


...



--- Address: 80 JESSE HILL JR DR SE, ATLANTA, GA 303033050 ---


,provider_full_name,primary_taxonomy_description,primary_practice_address_1,primary_practice_city,primary_practice_postal_code
34,NADIRAH ABDUL-MATEEN,"Nurse Practitioner, Family",80 JESSE HILL JR DR SE,ATLANTA,303033050
35,JOSEPH ABERGEL,Student in an Organized Health Care Education/...,80 JESSE HILL JR DR SE,ATLANTA,303033050




--- Address: 3495 PIEDMONT ROAD NE, NINE PIEDMONT CENTER, ATLANTA, GA 30305 ---


,provider_full_name,primary_taxonomy_description,primary_practice_address_1,primary_practice_city,primary_practice_postal_code
38,SALPI ADROUNY,Family Medicine,3495 PIEDMONT ROAD NE,ATLANTA,30305
39,LYDIA ALEXANDER,Internal Medicine,3495 PIEDMONT ROAD NE,ATLANTA,30305
40,RICHARD ABROHAMS,"Internal Medicine, Geriatric Medicine",3495 PIEDMONT ROAD NE,ATLANTA,30305
41,ANDREW ADAM,Pediatrics,3495 PIEDMONT ROAD NE,ATLANTA,30305




Total unique shared practice locations with more than one physician: 66
Total physicians in shared practice locations: 416
